# SMG_Lite — experiments that fill `result_word.docx`

Every experiment is **one small cell**: one model × one dataset (train → pick best epoch on
validation → test once on the held-out test split). Results are saved in `outputs/`, so

* a finished cell is **skipped** when re-run (set `force=True` in the config to retrain),
* an interrupted cell (Colab / Kaggle time-out) **resumes** from its last epoch,
* the last section **combines** everything into the tables, figures and a filled copy of
  `templates/result_word.docx` → `outputs/result_word_filled.docx`.

### Fast **and** reliable (Google Colab) — do this once
Upload 4 zip files to Google Drive, exactly here:
```
MyDrive/SMG_Lite/
├── SMG_Lite_Experiments.ipynb  ← this notebook (open it from here with Google Colaboratory)
├── SMG_Lite_Project.zip        ← this project folder, zipped
└── datasets/
    ├── plantvillage.zip        ← zip of the class folders
    ├── rice.zip
    └── mango.zip
```
Then open this notebook in Colab (Runtime → GPU) and run from the top. The first cells
1. copy the project and unzip the datasets to Colab's **local disk** → training is **fast**;
2. save every checkpoint and result locally **and copy it to `MyDrive/SMG_Lite/outputs/` after every epoch** → nothing is lost when Colab disconnects;
3. in a new session **restore** everything from Drive → finished cells are skipped, an interrupted cell continues from its last epoch.

On your own PC / Kaggle nothing changes: put the images in `data/<dataset>/` and run.

**Run order:** 0 (setup) → A (Plant Village) → B (Rice) → C (Mango Leaf) → D (Ablation) → E (Combine).
Section A must run before B/C, because B/C start each model from its own Plant Village
checkpoint (`transfer_from="plantvillage"`, applied identically to every model).

| Section | Cells | What it fills in the docx |
|---|---|---|
| A / B / C | 8 models + 2 no-augmentation runs per dataset | the 3 performance tables, metric line chart, training curves, confusion matrix, ROC |
| D | 8 ablation variants (own dataset + own epoch budget) | the ablation table |
| E | combine | `outputs/result_word_filled.docx` + all PNG / CSV files |

> The numbers in the tables are whatever your training runs produce — nothing is typed in by hand.

## 0. Setup

In [ ]:
# 0.1  Where am I?  Colab -> Drive for safety, local disk for speed.  PC / Kaggle -> this folder.
import os, sys, zipfile, subprocess
DRIVE_DIR = "/content/drive/MyDrive/SMG_Lite"   # Colab: folder on Google Drive (see above)
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/SMG_Lite_Project"     # local disk = fast
    if not os.path.exists(os.path.join(PROJECT_DIR, "smg_lab.py")):
        zipfile.ZipFile(os.path.join(DRIVE_DIR, "SMG_Lite_Project.zip")).extractall("/content")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "python-docx"])
else:
    PROJECT_DIR = "."                             # folder that contains SMG_Lite.py and smg_lab.py

os.chdir(PROJECT_DIR); sys.path.insert(0, os.getcwd())
print("project:", os.getcwd(), "| Colab:", IN_COLAB)

In [ ]:
# ------------------------------- CONFIG -------------------------------------
DATASETS = {
    #  key              folder with one sub-folder per class (or train/ val/ test/)   docx title     epochs
    "plantvillage": dict(path="data/plantvillage", title="Plant Village", epochs=30),
    "rice":         dict(path="data/rice",         title="Rice",          epochs=40),
    "mango":        dict(path="data/mango",        title="Mango Leaf",    epochs=40),
}
TABLE_ORDER      = ["plantvillage", "rice", "mango"]   # order of the 3 tables in result_word.docx

ABLATION_DATASET = "rice"   # ablation uses a different dataset ...
ABLATION_EPOCHS  = 20       # ... and its own epoch budget

QUICK_TEST = False          # True -> 1 epoch x 3 batches per cell, written to outputs_quicktest/

import smg_lab as lab
lab.configure(
    datasets=DATASETS,
    out_dir="outputs_quicktest" if QUICK_TEST else "outputs",
    split=(0.70, 0.15, 0.15),      # train / validation / test (stratified, seed fixed)
    seed=42,
    img_size=224, batch_size=64, workers=4,
    lr=3e-3, wd=5e-2, warmup=3,    # AdamW + warm-up + cosine (train.py)
    mixup=0.2, smooth=0.1, ema_decay=0.999,
    randaug_magnitude=7,
    mixup_off_frac=0.2,            # MixUp/CutMix off for the last 20 % of epochs
    tta=True,                      # flip test-time augmentation, same for every model
    lr_override={"VGG16": 1e-4},   # plain VGG16 (no BatchNorm) diverges at 3e-3 from scratch
    transfer_from="plantvillage",  # Rice / Mango runs start from the same model's Plant Village weights
    ablation_epochs=ABLATION_EPOCHS,
    force=False,
    max_batches=3 if QUICK_TEST else None,
    # Colab: every result + checkpoint is copied to Drive after each epoch, and restored next session
    backup_dir=(os.path.join(DRIVE_DIR, "outputs_quicktest" if QUICK_TEST else "outputs")
                if IN_COLAB else None),
)
# Colab: unzip MyDrive/SMG_Lite/datasets/<name>.zip to the local disk (once per session)
lab.prepare_data(os.path.join(DRIVE_DIR, "datasets") if IN_COLAB else None)
if QUICK_TEST:
    for d in DATASETS.values():
        d["epochs"] = 1
    lab.CFG.update(ablation_epochs=1, warmup=0)
print("device:", lab.DEVICE, "| results ->", lab.CFG["out_dir"])

In [ ]:
# Sanity check: size / compute of every model (38 classes, 224x224) + the data splits
import pandas as pd
rows = []
for m in lab.MODEL_NAMES:
    c = lab.model_cost(lambda nc, m=m: lab.build_model(m, nc), 38)
    rows.append([lab.DISPLAY[m], f"{c['params']:,}", f"{c['size_mb']:.2f}", f"{c['gflops']:.3f}"])
display(pd.DataFrame(rows, columns=["Model", "Trainable parameters", "Size (MB)", "GFLOPs"]))

for name, d in DATASETS.items():
    if os.path.isdir(d["path"]):
        s = lab.get_split(name)
        print(f"{name:13s} classes={len(s['classes']):3d}  train={len(s['train']):6d}  "
              f"val={len(s['val']):5d}  test={len(s['test']):5d}")
    else:
        print(f"{name:13s} -> folder {d['path']} not found (put the dataset there)")

## A. Plant Village — table "Performance comparison of various Models on the Plant Village dataset"

In [ ]:
# A1 — MobileNet V3 Large on Plant Village
lab.run("plantvillage", "MobileNetV3-Large")

In [ ]:
# A2 — MobileNet V2 on Plant Village
lab.run("plantvillage", "MobileNetV2")

In [ ]:
# A3 — Standard VGG16 on Plant Village
lab.run("plantvillage", "VGG16")

In [ ]:
# A4 — ResNet50 on Plant Village
lab.run("plantvillage", "ResNet50")

In [ ]:
# A5 — InceptionV3 on Plant Village
lab.run("plantvillage", "InceptionV3")

In [ ]:
# A6 — Xception on Plant Village
lab.run("plantvillage", "Xception")

In [ ]:
# A7 — DenseNet121 on Plant Village
lab.run("plantvillage", "DenseNet121")

In [ ]:
# A8 — SMG_Lite (proposed) on Plant Village
lab.run("plantvillage", "SMG_Lite")

**Plant Village: runs without augmentation** — only needed for the training-curve figure (MobileNetV2 vs proposed, with and without augmentation).

In [ ]:
# A9 — MobileNet V2 on Plant Village, NO augmentation
lab.run("plantvillage", "MobileNetV2", aug=False)

In [ ]:
# A10 — SMG_Lite on Plant Village, NO augmentation
lab.run("plantvillage", "SMG_Lite", aug=False)

## B. Rice — table "Performance comparison of various Models on the Rice dataset"
Each model starts from its own Plant Village checkpoint (section A), then trains on this dataset.

In [ ]:
# B1 — MobileNet V3 Large on Rice
lab.run("rice", "MobileNetV3-Large")

In [ ]:
# B2 — MobileNet V2 on Rice
lab.run("rice", "MobileNetV2")

In [ ]:
# B3 — Standard VGG16 on Rice
lab.run("rice", "VGG16")

In [ ]:
# B4 — ResNet50 on Rice
lab.run("rice", "ResNet50")

In [ ]:
# B5 — InceptionV3 on Rice
lab.run("rice", "InceptionV3")

In [ ]:
# B6 — Xception on Rice
lab.run("rice", "Xception")

In [ ]:
# B7 — DenseNet121 on Rice
lab.run("rice", "DenseNet121")

In [ ]:
# B8 — SMG_Lite (proposed) on Rice
lab.run("rice", "SMG_Lite")

**Rice: runs without augmentation** — only needed for the training-curve figure (MobileNetV2 vs proposed, with and without augmentation).

In [ ]:
# B9 — MobileNet V2 on Rice, NO augmentation
lab.run("rice", "MobileNetV2", aug=False)

In [ ]:
# B10 — SMG_Lite on Rice, NO augmentation
lab.run("rice", "SMG_Lite", aug=False)

## C. Mango Leaf — table "Performance comparison of various Models on the Mango Leaf dataset"
Each model starts from its own Plant Village checkpoint (section A), then trains on this dataset.

In [ ]:
# C1 — MobileNet V3 Large on Mango Leaf
lab.run("mango", "MobileNetV3-Large")

In [ ]:
# C2 — MobileNet V2 on Mango Leaf
lab.run("mango", "MobileNetV2")

In [ ]:
# C3 — Standard VGG16 on Mango Leaf
lab.run("mango", "VGG16")

In [ ]:
# C4 — ResNet50 on Mango Leaf
lab.run("mango", "ResNet50")

In [ ]:
# C5 — InceptionV3 on Mango Leaf
lab.run("mango", "InceptionV3")

In [ ]:
# C6 — Xception on Mango Leaf
lab.run("mango", "Xception")

In [ ]:
# C7 — DenseNet121 on Mango Leaf
lab.run("mango", "DenseNet121")

In [ ]:
# C8 — SMG_Lite (proposed) on Mango Leaf
lab.run("mango", "SMG_Lite")

**Mango Leaf: runs without augmentation** — only needed for the training-curve figure (MobileNetV2 vs proposed, with and without augmentation).

In [ ]:
# C9 — MobileNet V2 on Mango Leaf, NO augmentation
lab.run("mango", "MobileNetV2", aug=False)

In [ ]:
# C10 — SMG_Lite on Mango Leaf, NO augmentation
lab.run("mango", "SMG_Lite", aug=False)

## D. Ablation studies on SMG_Lite

Runs on `ABLATION_DATASET` with `ABLATION_EPOCHS` (set in the config), every variant from scratch.
`SMG_Lite.py` is not edited — components are switched off when the model is built:

* **Ghost ✗** → ordinary 1×1 expansion conv instead of the Ghost expansion
* **MS-DW gate ✗** → a single depthwise 3×3 instead of the gated 3×3 + 5×5 multi-scale depthwise
* **CEA ✗** → the Coordinate-ECA attention is removed (identity)

To run the ablation on another dataset as well, change `ABLATION_DATASET` and run these 8 cells again;
section E then builds one ablation table per dataset.

In [ ]:
# D1 — Ghost ✗ | MS-DW gate ✗ | CEA ✗
lab.run_ablation(ABLATION_DATASET, ghost=False, msdw=False, cea=False)

In [ ]:
# D2 — Ghost ✓ | MS-DW gate ✗ | CEA ✗
lab.run_ablation(ABLATION_DATASET, ghost=True, msdw=False, cea=False)

In [ ]:
# D3 — Ghost ✗ | MS-DW gate ✓ | CEA ✗
lab.run_ablation(ABLATION_DATASET, ghost=False, msdw=True, cea=False)

In [ ]:
# D4 — Ghost ✗ | MS-DW gate ✗ | CEA ✓
lab.run_ablation(ABLATION_DATASET, ghost=False, msdw=False, cea=True)

In [ ]:
# D5 — Ghost ✓ | MS-DW gate ✓ | CEA ✗
lab.run_ablation(ABLATION_DATASET, ghost=True, msdw=True, cea=False)

In [ ]:
# D6 — Ghost ✓ | MS-DW gate ✗ | CEA ✓
lab.run_ablation(ABLATION_DATASET, ghost=True, msdw=False, cea=True)

In [ ]:
# D7 — Ghost ✗ | MS-DW gate ✓ | CEA ✓
lab.run_ablation(ABLATION_DATASET, ghost=False, msdw=True, cea=True)

In [ ]:
# D8 — Ghost ✓ | MS-DW gate ✓ | CEA ✓
lab.run_ablation(ABLATION_DATASET, ghost=True, msdw=True, cea=True)

## E. Combine → tables, figures and `result_word_filled.docx`

In [ ]:
ablation_sets = [d for d in DATASETS
                 if os.path.isdir(os.path.join(lab.CFG["out_dir"], "ablation", d))] or [ABLATION_DATASET]
report = lab.make_report(TABLE_ORDER, ablation_sets, template="templates/result_word.docx")

In [ ]:
# Show every figure in the same order as result_word.docx
from IPython.display import Image, display
out = lab.CFG["out_dir"]
for prefix in ["figures/metrics_lines", "figures/curves", "figures/confusion", "figures/roc", "tables/ablation"]:
    for ds in TABLE_ORDER + [d for d in DATASETS if d not in TABLE_ORDER]:
        f = os.path.join(out, f"{prefix}_{ds}.png")
        if os.path.exists(f):
            print(f); display(Image(f, width=900))